In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.sparse import csr_matrix

pd.set_option('display.max_rows', 20)
pd.set_option('display.max_columns', 20)

In [2]:
# Load processed data (created by data_prep.ipynb)
processed_dir = Path('../data/processed')
ratings_path = processed_dir / 'ratings_filtered.csv'

if not ratings_path.exists():
    raise FileNotFoundError('Processed files not found. Run the save cell in data_prep.ipynb first.')

ratings_f = pd.read_csv(ratings_path)
print('ratings_f', ratings_f.shape)

ratings_f (31498689, 6)


In [3]:
# Build user/item indices
user_ids = ratings_f['userId'].unique()
movie_ids = ratings_f['movieId'].unique()

user_id_to_idx = {uid: i for i, uid in enumerate(user_ids)}
movie_id_to_idx = {mid: i for i, mid in enumerate(movie_ids)}

ratings_f['user_idx'] = ratings_f['userId'].map(user_id_to_idx)
ratings_f['movie_idx'] = ratings_f['movieId'].map(movie_id_to_idx)

print('Index ranges:', ratings_f['user_idx'].max(), ratings_f['movie_idx'].max())

Index ranges: 200946 16033


In [4]:
# Create implicit feedback (like if rating >= 4.0)
ratings_f['implicit'] = (ratings_f['rating'] >= 4.0).astype(np.float32)

# Keep only positive interactions for implicit ALS
likes = ratings_f[ratings_f['implicit'] == 1.0].copy()
print('Positive interactions:', likes.shape)

Positive interactions: (15799204, 7)


In [6]:
# Leave-one-out split per user (hold out most recent like)
if pd.api.types.is_numeric_dtype(likes['timestamp']):
    likes['timestamp'] = pd.to_datetime(likes['timestamp'], unit='s')
else:
    likes['timestamp'] = pd.to_datetime(likes['timestamp'], errors='coerce')
likes = likes.sort_values(['user_idx', 'timestamp'])

test_idx = likes.groupby('user_idx').tail(1).index
test = likes.loc[test_idx]
train = likes.drop(test_idx)

print('Train likes:', train.shape)
print('Test likes:', test.shape)
print('Users in train:', train['user_idx'].nunique())
print('Users in test:', test['user_idx'].nunique())

Train likes: (15598480, 7)
Test likes: (200724, 7)
Users in train: 200462
Users in test: 200724


In [7]:
# Build sparse user-item matrix for ALS
n_users = len(user_ids)
n_movies = len(movie_ids)

X = csr_matrix(
    (train['implicit'], (train['user_idx'], train['movie_idx'])),
    shape=(n_users, n_movies)
)

print('Train matrix shape:', X.shape)
print('Non-zeros:', X.nnz)

Train matrix shape: (200947, 16034)
Non-zeros: 15598480


In [9]:
# Install implicit if needed (run once)
!pip install implicit

     |████████████████████████████████| 645 kB 3.4 MB/s eta 0:00:01
     |████████████████████████████████| 78 kB 5.1 MB/s eta 0:00:011
You should consider upgrading via the '/Users/mironshoh/Desktop/workspaces/projects/personal/CineMatch/.venv/bin/python3 -m pip install --upgrade pip' command.


In [10]:
from implicit.als import AlternatingLeastSquares

/Users/mironshoh/Desktop/workspaces/projects/personal/CineMatch/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
# Hyperparameter grid
factors_list = [64, 128, 256]
reg_list = [0.01, 0.05, 0.1]
alpha_list = [20, 40, 80]

def precision_recall_map_at_k_als(model, test_df, k=10):
    # Evaluate using leave-one-out test set (one like per user)
    test_items = test_df.groupby('user_idx')['movie_idx'].apply(set).to_dict()
    users = list(test_items.keys())

    precisions = []
    recalls = []
    ap_scores = []

    user_items = train.groupby('user_idx')['movie_idx'].apply(set).to_dict()

    for user_idx in users:
        relevant = test_items.get(user_idx, set())
        if not relevant:
            continue

        recs = model.recommend(user_idx, X[user_idx], N=k, filter_already_liked_items=True)
        if isinstance(recs, tuple) and len(recs) == 2:
            recs = list(recs[0])
        else:
            recs = [item for item, _ in recs]

        hits = [1 if r in relevant else 0 for r in recs]
        precision = sum(hits) / k
        recall = sum(hits) / len(relevant)

        cum_hits = 0
        ap = 0.0
        for i, h in enumerate(hits, start=1):
            if h:
                cum_hits += 1
                ap += cum_hits / i
        ap = ap / min(len(relevant), k)

        precisions.append(precision)
        recalls.append(recall)
        ap_scores.append(ap)

    return {
        'precision@k': float(np.mean(precisions)) if precisions else 0.0,
        'recall@k': float(np.mean(recalls)) if recalls else 0.0,
        'map@k': float(np.mean(ap_scores)) if ap_scores else 0.0,
        'users_evaluated': len(precisions)
    }

results = []
for factors in factors_list:
    for reg in reg_list:
        for alpha in alpha_list:
            model = AlternatingLeastSquares(factors=factors, regularization=reg, iterations=20)
            model.fit(X * alpha)
            metrics = precision_recall_map_at_k_als(model, test, k=10)
            results.append({
                'factors': factors,
                'regularization': reg,
                'alpha': alpha,
                **metrics
            })
            print(results[-1])

results_df = pd.DataFrame(results).sort_values(['map@k', 'precision@k', 'recall@k'], ascending=False)
results_df.head(5)

100%|██████████| 20/20 [01:18<00:00,  3.92s/it]


{'factors': 64, 'regularization': 0.01, 'alpha': 20, 'precision@k': 0.009612702018692335, 'recall@k': 0.09612702018692333, 'map@k': 0.032854238292302496, 'users_evaluated': 200724}


100%|██████████| 20/20 [01:17<00:00,  3.89s/it]


{'factors': 64, 'regularization': 0.01, 'alpha': 40, 'precision@k': 0.00888284410434228, 'recall@k': 0.08882844104342281, 'map@k': 0.02952777611712269, 'users_evaluated': 200724}


100%|██████████| 20/20 [01:13<00:00,  3.68s/it]


...

100%|██████████| 20/20 [10:15<00:00, 30.76s/it]


{'factors': 256, 'regularization': 0.1, 'alpha': 40, 'precision@k': 0.010112891333373189, 'recall@k': 0.10112891333373189, 'map@k': 0.035905282796910104, 'users_evaluated': 200724}


100%|██████████| 20/20 [11:56<00:00, 35.82s/it]


{'factors': 256, 'regularization': 0.1, 'alpha': 80, 'precision@k': 0.009802514896076205, 'recall@k': 0.09802514896076205, 'map@k': 0.034455756273401396, 'users_evaluated': 200724}


,factors,regularization,alpha,precision@k,recall@k,map@k,users_evaluated
24,256,0.10,20,0.010322,0.103221,0.037534,200724
21,256,0.05,20,0.010231,0.102315,0.037222,200724
18,256,0.01,20,0.010128,0.101278,0.036776,200724
22,256,0.05,40,0.010138,0.101378,0.035947,200724
25,256,0.10,40,0.010113,0.101129,0.035905,200724


In [13]:
results_df

,factors,regularization,alpha,precision@k,recall@k,map@k,users_evaluated
24,256,0.10,20,0.010322,0.103221,0.037534,200724
21,256,0.05,20,0.010231,0.102315,0.037222,200724
18,256,0.01,20,0.010128,0.101278,0.036776,200724
22,256,0.05,40,0.010138,0.101378,0.035947,200724
25,256,0.10,40,0.010113,0.101129,0.035905,200724
...,...,...,...,...,...,...,...
14,128,0.05,80,0.008904,0.089038,0.029368,200724
11,128,0.01,80,0.008853,0.088535,0.029314,200724
5,64,0.05,80,0.008114,0.081141,0.026382,200724
2,64,0.01,80,0.008000,0.080000,0.025990,200724
